# Week 2: AI Foundations - Live Session
## From Secrets to Attention: Building Your AI Intuition

**Student Exercises Notebook**

Use this notebook to work through the same flow as the master notebook, but fill in the code cells yourself using the hints provided.

# Topic 1: Environment Variables & Secrets ??

**Objective**: Learn why `.env` files matter, how to store secrets securely, and access them in code.

In [ ]:
# Exercise 1: Load environment variables from .env
import os
from dotenv import load_dotenv

load_dotenv()


def mask_secret(value: str | None, keep: int = 4) -> str:
    if not value:
        return "Not set"
    if len(value) <= keep:
        return "*" * len(value)
    return f"{'*' * (len(value) - keep)}{value[-keep:]}"


openai_api_key = os.getenv("OPENAI_API_KEY")
print("OPENAI_API_KEY status:", "Set" if openai_api_key else "Not set")
print("Masked preview:", mask_secret(openai_api_key))

### Exercise: Create and check `TEST_SECRET`

Set `TEST_SECRET` in your `.env` file, then write code to confirm whether it is available.

In [ ]:
# Exercise 2: Check TEST_SECRET
import os


def mask_secret(value: str | None, keep: int = 3) -> str:
    if not value:
        return "Not set"
    if len(value) <= keep:
        return "*" * len(value)
    return f"{'*' * (len(value) - keep)}{value[-keep:]}"


test_secret = os.getenv("TEST_SECRET")
print("TEST_SECRET status:", "Set" if test_secret else "Not set")
print("Masked preview:", mask_secret(test_secret))

### Practice: Check multiple variables

Try reading `OPENAI_API_KEY`, `HF_API_KEY`, and `LANGCHAIN_API_KEY`, then print whether each one is set.

In [ ]:
# Exercise 3: Check several environment variables
import os

var_names = ["OPENAI_API_KEY", "HF_API_KEY", "LANGCHAIN_API_KEY"]
for name in var_names:
    value = os.getenv(name)
    print(f"{name}: {'Set' if value else 'Not set'}")

---
# Topic 2: OpenAI Library & LLM Integration ??

**Objective**: Learn to use the OpenAI Python library to interact with LLMs via the Chat API.

In [ ]:
# Exercise 4: Import OpenAI and prepare a client
import os
from openai import OpenAI

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY is not set. Add it to your .env file first.")

client = OpenAI(api_key=api_key)
chat_model = "gpt-4o-mini"
embedding_model = "text-embedding-3-small"

print("OpenAI client initialized successfully.")

## Chat API Fundamentals

The Chat API uses `messages` with roles such as `system`, `user`, and `assistant`.

In [ ]:
# Exercise 5: Make your first chat call
response_ex5 = client.chat.completions.create(
    model=chat_model,
    messages=[
        {"role": "system", "content": "You are a helpful teaching assistant."},
        {"role": "user", "content": "Explain tokenization in one short paragraph."},
    ],
    max_tokens=80,
    temperature=0.3,
)

print("Assistant:", response_ex5.choices[0].message.content)
if getattr(response_ex5, "usage", None):
    print("Usage:", response_ex5.usage)

### Temperature Exercise

Try different temperature values such as `0.0`, `0.7`, and `1.5` and compare the outputs.

In [ ]:
# Exercise 6: Compare responses at different temperatures
prompt = "Write a one-line analogy for how attention works in transformers."
for temp in [0.0, 0.7, 1.5]:
    resp = client.chat.completions.create(
        model=chat_model,
        messages=[{"role": "user", "content": prompt}],
        temperature=temp,
        max_tokens=60,
    )
    print(f"\nTemperature = {temp}")
    print(resp.choices[0].message.content)

## System Prompt Demo

Use a system prompt to change the assistant style or behavior.

In [ ]:
# Exercise 7: Test different system prompts
user_prompt = "Explain embeddings in 2 to 3 sentences."
system_prompts = [
    "You are a strict teacher. Be precise and formal.",
    "You are a friendly mentor. Use simple language.",
    "You are concise. Use only bullet points.",
]

for sys_prompt in system_prompts:
    resp = client.chat.completions.create(
        model=chat_model,
        messages=[
            {"role": "system", "content": sys_prompt},
            {"role": "user", "content": user_prompt},
        ],
        max_tokens=100,
        temperature=0.5,
    )
    print(f"\nSystem Prompt: {sys_prompt}")
    print(resp.choices[0].message.content)

## Hallucinations

Try prompts that ask for uncertain or future events, then observe how the model responds.

In [ ]:
# Exercise 8: Explore hallucination-prone prompts
hallucination_prompts = [
    "Who will win the FIFA World Cup in 2038?",
    "Give exact stock prices for tomorrow.",
    "What discoveries will be made in AI in the year 2050?",
]

for p in hallucination_prompts:
    resp = client.chat.completions.create(
        model=chat_model,
        messages=[
            {"role": "system", "content": "Be honest about uncertainty and avoid making up facts."},
            {"role": "user", "content": p},
        ],
        temperature=0.4,
        max_tokens=120,
    )
    print(f"\nPrompt: {p}")
    print(resp.choices[0].message.content)

## Interactive Input

Write a small prompt that takes user input and sends it to the model.

In [ ]:
# Exercise 9: Accept user input and send it to the model
user_text = input("Ask anything: ").strip()
if not user_text:
    print("Please enter a non-empty question.")
else:
    response_ex9 = client.chat.completions.create(
        model=chat_model,
        messages=[{"role": "user", "content": user_text}],
        max_tokens=120,
    )
    print("Assistant:", response_ex9.choices[0].message.content)

## Token Count Awareness

Check the `usage` information returned by the API and note how many tokens were used.

In [ ]:
# Exercise 10: Print token usage from the last response
if "response_ex9" in globals():
    last_response = response_ex9
elif "response_ex5" in globals():
    last_response = response_ex5
else:
    last_response = None

if last_response is None:
    print("No response object found. Run Exercise 5 or 9 first.")
elif getattr(last_response, "usage", None):
    usage = last_response.usage
    print("prompt_tokens:", getattr(usage, "prompt_tokens", None))
    print("completion_tokens:", getattr(usage, "completion_tokens", None))
    print("total_tokens:", getattr(usage, "total_tokens", None))
else:
    print("Usage information not available on this response.")

## Part 2.5: Vision Capabilities

Write a request that sends an image URL along with a text question.

In [ ]:
# Exercise 11: Try an image analysis request
image_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/d/dd/Gfp-wisconsin-madison-the-nature-boardwalk.jpg/640px-Gfp-wisconsin-madison-the-nature-boardwalk.jpg"

vision_response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "Describe this image in 2 short sentences."},
                {"type": "image_url", "image_url": {"url": image_url}},
            ],
        }
    ],
    max_tokens=100,
)

print(vision_response.choices[0].message.content)

---
# Topic 3: Tokenization Deep Dive ??

**Objective**: Understand how LLMs break down text into tokens, count them, and estimate costs.

In [ ]:
# Exercise 12: Load a tokenizer
import tiktoken

try:
    tokenizer = tiktoken.encoding_for_model(chat_model)
except KeyError:
    tokenizer = tiktoken.get_encoding("cl100k_base")

print("Tokenizer loaded:", tokenizer.name)

In [ ]:
# Exercise 13: Tokenize a sample sentence
sample_text = "Transformers process text by turning words into tokens."
tokens = tokenizer.encode(sample_text)

print("Text:", sample_text)
print("Tokens:", tokens)
print("Token count:", len(tokens))
print("Decoded:", tokenizer.decode(tokens))

### Exercise: Tokenize your own sentence

Pick any sentence, tokenize it, decode each token, and observe which words split into multiple tokens.

In [ ]:
# Exercise 14: Tokenize custom text
custom_text = "Unbelievability in tokenization can be counterintuitive."
custom_tokens = tokenizer.encode(custom_text)

print("Custom text:", custom_text)
print("Token count:", len(custom_tokens))
print("Token ids:", custom_tokens)
print("\nDecoded token-by-token:")
for token_id in custom_tokens:
    print(f"{token_id}: {repr(tokenizer.decode([token_id]))}")

### Cost Calculation

Estimate how token count affects the cost of a longer piece of text.

In [ ]:
# Exercise 15: Estimate cost from token count
long_text = ("RAG systems combine retrieval and generation. " * 400).strip()
long_token_count = len(tokenizer.encode(long_text))

# Example input pricing in USD per 1K tokens (update based on current provider pricing).
price_per_1k_tokens = 0.00015
estimated_cost = (long_token_count / 1000) * price_per_1k_tokens

print("Estimated tokens:", long_token_count)
print(f"Estimated input cost: ${estimated_cost:.6f} USD")

---
# Topic 4: Embeddings & Semantic Search ??

**Objective**: Learn what embeddings are, how to create them, and build a simple semantic search.

In [ ]:
# Exercise 16: Create embeddings for a few texts
import numpy as np

documents = [
    "Football is a popular sport played worldwide.",
    "Neural networks are used in deep learning.",
    "Cricket matches can last several hours.",
    "Transformers are powerful models for language tasks.",
    "Healthy food supports long-term well-being.",
    "Regular exercise improves cardiovascular health.",
]

embedding_response = client.embeddings.create(model=embedding_model, input=documents)
doc_embeddings = np.array([item.embedding for item in embedding_response.data], dtype=float)

print("Number of documents:", len(documents))
print("Embedding shape:", doc_embeddings.shape)

In [ ]:
# Exercise 17: Compute cosine similarity
import numpy as np


def cosine_similarity(vec_a: np.ndarray, vec_b: np.ndarray) -> float:
    denom = np.linalg.norm(vec_a) * np.linalg.norm(vec_b)
    if denom == 0:
        return 0.0
    return float(np.dot(vec_a, vec_b) / denom)

sim_0_2 = cosine_similarity(doc_embeddings[0], doc_embeddings[2])
sim_1_3 = cosine_similarity(doc_embeddings[1], doc_embeddings[3])

print(f"Similarity (sports docs): {sim_0_2:.4f}")
print(f"Similarity (AI docs): {sim_1_3:.4f}")

In [ ]:
# Exercise 18: Visualize embeddings in 2D
import matplotlib.pyplot as plt

# PCA with NumPy (no external ML dependency needed).
X = doc_embeddings - doc_embeddings.mean(axis=0, keepdims=True)
_, _, vt = np.linalg.svd(X, full_matrices=False)
components_2d = X @ vt[:2].T

plt.figure(figsize=(8, 5))
for i, label in enumerate(documents):
    x, y = components_2d[i]
    plt.scatter(x, y)
    plt.text(x + 0.01, y + 0.01, f"Doc {i}", fontsize=9)

plt.title("2D Projection of Document Embeddings")
plt.xlabel("PC 1")
plt.ylabel("PC 2")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Exercise 19: Build semantic search
query = "How are transformers used in natural language processing?"
query_embedding = np.array(
    client.embeddings.create(model=embedding_model, input=[query]).data[0].embedding,
    dtype=float,
)

scores = [cosine_similarity(query_embedding, emb) for emb in doc_embeddings]
ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)

print("Query:", query)
print("\nTop matches:")
for idx, score in ranked[:3]:
    print(f"- Score={score:.4f} | {documents[idx]}")

### Exercise: Build your own semantic search

Use your own document list, create embeddings, and try a query from a different topic.

In [ ]:
# Exercise 20: Semantic search with your own documents
my_docs = [
    "I enjoy building machine learning pipelines.",
    "Baking bread requires patience and timing.",
    "LangChain helps orchestrate LLM workflows.",
    "Strength training improves muscle endurance.",
    "Prompt engineering can improve model output quality.",
    "Travel photography captures stories from places.",
]

my_embeddings = np.array(
    [item.embedding for item in client.embeddings.create(model=embedding_model, input=my_docs).data],
    dtype=float,
)

my_query = "How can I improve outputs from language models?"
my_query_embedding = np.array(
    client.embeddings.create(model=embedding_model, input=[my_query]).data[0].embedding,
    dtype=float,
)

my_scores = [cosine_similarity(my_query_embedding, emb) for emb in my_embeddings]
my_ranked = sorted(enumerate(my_scores), key=lambda x: x[1], reverse=True)

print("Query:", my_query)
for idx, score in my_ranked[:3]:
    print(f"- Score={score:.4f} | {my_docs[idx]}")

---
# Topic 5: Attention Mechanism Explained ??

**Objective**: Understand how transformers focus on relevant parts of text using attention.

In [ ]:
# Exercise 21: Visualize attention weights
words = ["The", "cat", "sat", "on", "the", "mat"]
focus_word = "sat"
weights = [0.05, 0.25, 0.35, 0.15, 0.05, 0.15]  # sums to 1.0

print(f"Focus word: {focus_word}\n")
for w, score in zip(words, weights):
    bar = "#" * int(score * 40)
    print(f"{w:>5} | {bar:<40} {score:.2f}")

In [ ]:
# Exercise 22: Break down Query-Key-Value
rows = [
    ("Query (Q)", "What am I looking for in other words?"),
    ("Key (K)", "What kind of information does this word represent?"),
    ("Value (V)", "What information should be passed forward if selected?"),
]

print(f"{'Component':<12} | Meaning")
print("-" * 65)
for name, meaning in rows:
    print(f"{name:<12} | {meaning}")

In [ ]:
# Exercise 23: Implement scaled dot-product attention
import numpy as np


def softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:
    x_shifted = x - np.max(x, axis=axis, keepdims=True)
    exp_x = np.exp(x_shifted)
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)


def scaled_dot_product_attention(Q: np.ndarray, K: np.ndarray, V: np.ndarray):
    d_k = Q.shape[-1]
    scores = (Q @ K.T) / np.sqrt(d_k)
    weights = softmax(scores, axis=-1)
    output = weights @ V
    return output, weights

Q = np.array([[1.0, 0.0], [0.0, 1.0]])
K = np.array([[1.0, 0.2], [0.1, 1.0]])
V = np.array([[10.0, 1.0], [2.0, 8.0]])

attn_output, attn_weights = scaled_dot_product_attention(Q, K, V)
print("Attention weights:\n", attn_weights)
print("Attention output:\n", attn_output)

In [ ]:
# Exercise 24: Visualize an attention pattern
tokens = ["The", "cat", "sat", "on", "mat"]
attention_matrix = np.array([
    [0.40, 0.20, 0.15, 0.15, 0.10],
    [0.20, 0.35, 0.20, 0.15, 0.10],
    [0.10, 0.20, 0.40, 0.20, 0.10],
    [0.10, 0.15, 0.25, 0.35, 0.15],
    [0.10, 0.15, 0.20, 0.20, 0.35],
])

symbols = " .:-=+*#%@"

print("       " + "  ".join(f"{t:>3}" for t in tokens))
for i, row in enumerate(attention_matrix):
    shades = []
    for value in row:
        idx = min(int(value * (len(symbols) - 1)), len(symbols) - 1)
        shades.append(symbols[idx] * 3)
    print(f"{tokens[i]:>5}  " + " ".join(shades))

### Final Exercise: Trace attention manually

Choose a sentence, pick one focus word, and assign attention weights that sum to 1.0. Explain why certain words get higher or lower weights.

In [ ]:
# Exercise 25: Manually trace attention for a sentence
sentence_words = ["Large", "language", "models", "learn", "patterns"]
focus_word = "models"
manual_weights = np.array([0.10, 0.35, 0.30, 0.15, 0.10], dtype=float)

weight_sum = manual_weights.sum()
print("Focus word:", focus_word)
print(f"Sum of weights: {weight_sum:.2f}")

if not np.isclose(weight_sum, 1.0):
    print("Warning: weights should sum to 1.0")

print("\nAttention bars:")
for word, weight in zip(sentence_words, manual_weights):
    bar = "#" * int(weight * 50)
    print(f"{word:>10} | {bar:<50} {weight:.2f}")

---
# ?? Recap & Key Takeaways

## What We Covered

| Topic | Key Concept | Real-World Use |
|-------|-------------|----------------|
| **.env & Secrets** | Store sensitive data outside code | Production deployments, security |
| **OpenAI Library** | Chat API, functions, vision, streaming | Building AI applications |
| **Tokenization** | Text ? numbers, counting for cost | API budgeting, context management |
| **Embeddings** | Meaning ? vectors, semantic similarity | Search, recommendations, RAG |
| **Attention** | Focus mechanism, Q?K?V mechanism | How transformers work |

## Next Steps

1. Week 3: Build your first LangChain application
2. Week 4: Implement memory and agents
3. Week 5+: RAG systems and production deployments